# Data Mining – Lab 4
## Data Preparation: Data Quality, Cleaning, Integration, Transformation

**Dataset:** StudentsPerformance.csv  

**Лекцийн сэдэв:** Өгөгдөл бэлтгэх ба чанар (номын 2-р бүлэг)


### Lab зорилго
- Өгөгдлийн чанарыг (quality) шалгаж сурах
- Missing values болон noisy data-г цэвэрлэх аргуудыг практикт хэрэглэх
- Өгөгдөл нэгтгэх (integration), давхардал ба хамаарлыг (redundancy) илрүүлэх
- Normalization (min-max, z-score, decimal scaling) ба discretization хийх
- Lab 3-т үзсэн feature engineering-ийг бүрэн дүүрэн data preparation процесс болгон өргөтгөх

## 0) Dataset унших ба ерөнхий шалгалт

Өмнөх lab-уудын адил **StudentsPerformance.csv** dataset-ийг ашиглана.


In [1]:
%pip install numpy pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("StudentsPerformance.csv")
df.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group D,associate's degree,standard,completed,93,97,100
1,male,group B,master's degree,standard,none,72,51,63
2,female,group D,high school,standard,completed,53,66,64
3,female,group C,master's degree,standard,none,68,100,89
4,female,group D,bachelor's degree,standard,completed,100,100,100


In [3]:
print(df.shape)
df.info()

(1000, 8)
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   gender                       1000 non-null   str  
 1   race/ethnicity               1000 non-null   str  
 2   parental level of education  1000 non-null   str  
 3   lunch                        1000 non-null   str  
 4   test preparation course      1000 non-null   str  
 5   math score                   1000 non-null   int64
 6   reading score                1000 non-null   int64
 7   writing score                1000 non-null   int64
dtypes: int64(3), str(5)
memory usage: 62.6 KB


## 1) Өгөгдлийн чанар (Data Quality)

Өгөгдөл нь ашиглах зорилгодоо нийцэж байвал чанартай гэж үзнэ.  
Чанарын хүчин зүйлс: **accuracy, completeness, consistency, timeliness, believability, interpretability**.


### TASK 1: Quality check

1. Багана бүрийн дутуу утгын тоог ол (completeness).
2. Давхардсан мөрийн тоог ол.
3. Оноонууд 0–100 мужид байгаа эсэхийг шалга (accuracy).
4. Категори багануудын өвөрмөц утгуудыг харж, бичих хэлбэр зөрүүтэй эсэхийг шалга (consistency).


In [ ]:
missing_per_col = df.isna().sum()
print("Дутуу утга:\n", missing_per_col)

dup_count = df.duplicated().sum()
print("Давхардсан мөр:", dup_count)

score_cols = ['math score', 'reading score', 'writing score']
print(df[score_cols].agg(['min', 'max']))

for col in ['gender', 'lunch', 'test preparation course']:
    print(col, df[col].unique())

Дутуу утга:
 gender                         0
race/ethnicity                 0
parental level of education    0
lunch                          0
test preparation course        0
math score                     0
reading score                  0
writing score                  0
dtype: int64
Давхардсан мөр: 0
     math score  reading score  writing score
min           0             17             10
max         100            100            100
gender <StringArray>
['female', 'male']
Length: 2, dtype: str
lunch <StringArray>
['standard', 'free/reduced']
Length: 2, dtype: str
test preparation course <StringArray>
['completed', 'none']
Length: 2, dtype: str


## 2) Data Cleaning – Missing values

StudentsPerformance.csv нь цэвэр dataset тул бодит нөхцөлийг дууриахын тулд  
зарим утгыг **зориудаар дутуу** болгоно (math, reading score-ийн 10%, lunch-ийн 5%).


In [5]:
rng = np.random.default_rng(42)
df_miss = df.copy()

for col in ['math score', 'reading score']:
    df_miss[col] = df_miss[col].astype(float)
    idx = rng.choice(df_miss.index, size=int(0.10 * len(df_miss)), replace=False)
    df_miss.loc[idx, col] = np.nan

idx = rng.choice(df_miss.index, size=int(0.05 * len(df_miss)), replace=False)
df_miss.loc[idx, 'lunch'] = np.nan

df_miss.isnull().sum()

gender                           0
race/ethnicity                   0
parental level of education      0
lunch                           50
test preparation course          0
math score                     100
reading score                  100
writing score                    0
dtype: int64

### TASK 2: Дутуу утгын хувь

Багана бүрийн дутуу утгын **хувийг** ол.


In [ ]:
missing_pct = df_miss.isnull().mean() * 100
missing_pct

gender                          0.0
race/ethnicity                  0.0
parental level of education     0.0
lunch                           5.0
test preparation course         0.0
math score                     10.0
reading score                  10.0
writing score                   0.0
dtype: float64

### TASK 3: Missing values нөхөх аргууд

Дараах 5 аргыг хэрэглэ:

- a) Дутуу утгатай мөрийг орхих (ignore tuple)
- b) Баганын **mean**-ээр нөхөх
- c) Баганын **median**-аар нөхөх
- d) Ижил **class**-ийн mean-ээр нөхөх (gender-ээр бүлэглэнэ)
- e) Категори баганыг хамгийн их тохиолдох утгаар (**mode**) нөхөх


In [ ]:
df_a = df_miss.dropna()
print("a) үлдсэн мөр:", len(df_a))

df_b = df_miss.copy()
df_b['math score'] = df_b['math score'].fillna(df_b['math score'].mean())

df_c = df_miss.copy()
df_c['reading score'] = df_c['reading score'].fillna(df_c['reading score'].median())

df_d = df_miss.copy()
df_d['math score'] = df_d['math score'].fillna(
    df_d.groupby('gender')['math score'].transform('mean')
)

df_e = df_miss.copy()
df_e['lunch'] = df_e['lunch'].fillna(df_e['lunch'].mode()[0])

print(df_b['math score'].isnull().sum(),
      df_d['math score'].isnull().sum(),
      df_e['lunch'].isnull().sum())

a) үлдсэн мөр: 772
0 0 0


### TASK 4: Аргуудыг харьцуулах

Анхны (дутуу утгагүй) `math score`-ын mean, std-ийг нөхсөн аргуудтай харьцуул.


In [ ]:
comparison = pd.DataFrame({
    'original': df['math score'].agg(['mean', 'std']),
    'ignore_tuple': df_a['math score'].agg(['mean', 'std']),
    'mean_fill': df_b['math score'].agg(['mean', 'std']),
    'class_mean_fill': df_d['math score'].agg(['mean', 'std'])
})
comparison

,original,ignore_tuple,mean_fill,class_mean_fill
mean,70.105000,70.150259,70.212222,70.236315
std,14.468009,14.304842,13.481774,13.495603


### TASK 4.1: Тайлбар (үгээр)

1. Аль арга std-ийг хамгийн ихээр багасгав? Яагаад?
2. Mean-ээр нөхөх нь өгөгдлийг хэрхэн хазайлгаж болох вэ?
3. Хэзээ "tuple орхих" арга зохимжтой вэ?


## 3) Data Cleaning – Noisy data

Noise нь хэмжсэн утга дахь санамсаргүй алдаа эсвэл хэлбэлзэл юм.  
Зөөллөх аргууд: **binning**, **regression**, **outlier analysis**.


### TASK 5: Binning – зөөллөх

Лекцийн жишээ: эрэмбэлсэн үнэ `4, 8, 15, 21, 21, 24, 25, 28, 34`.

1. Өгөгдлийг 3 bin-д (bin бүрт 3 утга) хуваа (equal-frequency).
2. **Bin means**-ээр зөөллө.
3. **Bin boundaries**-аар зөөллө.


In [ ]:
prices = pd.Series([4, 8, 15, 21, 21, 24, 25, 28, 34])

bins = [prices[i:i+3] for i in range(0, len(prices), 3)]

smooth_mean = [[round(b.mean())] * len(b) for b in bins]

smooth_bound = []
for b in bins:
    lo, hi = b.min(), b.max()
    smooth_bound.append([int(lo) if (v - lo) <= (hi - v) else int(hi) for v in b])

print("Bins       :", [list(b) for b in bins])
print("By means   :", smooth_mean)
print("By bounds  :", smooth_bound)

Bins       : [[4, 8, 15], [21, 21, 24], [25, 28, 34]]
By means   : [[9, 9, 9], [22, 22, 22], [29, 29, 29]]
By bounds  : [[4, 4, 15], [21, 21, 24], [25, 25, 34]]


### TASK 6: Outlier илрүүлэх – IQR арга

`math score` дээр IQR аргаар outlier ол.

- Q1 = 25-р percentile, Q3 = 75-р percentile  
- IQR = Q3 − Q1  
- Outlier: `< Q1 − 1.5·IQR` эсвэл `> Q3 + 1.5·IQR`


In [ ]:
q1 = df['math score'].quantile(0.25)
q3 = df['math score'].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

outliers = df[(df['math score'] < lower) | (df['math score'] > upper)]
print("lower:", lower, "upper:", upper, "outlier тоо:", len(outliers))
outliers[['math score']].head()

lower: 30.0 upper: 110.0 outlier тоо: 8


,math score
59,0
76,8
211,18
316,22
338,23


### TASK 7: Outlier илрүүлэх – z-score арга

Mean-аас **2σ**-аас хол утгыг сэжигтэй outlier гэж үзье.


In [ ]:
z = (df['math score'] - df['math score'].mean()) / df['math score'].std()
z_outliers = df[z.abs() > 2]
print("z-score outlier тоо:", len(z_outliers))

z-score outlier тоо: 43


### TASK 7.1: Тайлбар (үгээр)

1. IQR болон z-score аргаар олдсон outlier-ийн тоо ялгаатай байна уу? Яагаад?
2. Оноо 0 байх нь "алдаа" юм уу, эсвэл бодит утга уу? Outlier-ийг устгахаас өмнө юуг шалгах ёстой вэ?


## 4) Data Integration – Өгөгдөл нэгтгэх

Олон эх сурвалжийн өгөгдлийг нэг санд нэгтгэхэд **entity identification**, **redundancy**,  
**tuple duplication**, **data value conflict** асуудлууд гарна.  

Бодит нөхцөлийг дууриахын тулд dataset-ийг хоёр хүснэгтэд хувааж, давхардал нэмнэ.


In [ ]:
students = df[['gender', 'race/ethnicity', 'parental level of education',
               'lunch', 'test preparation course']].copy()
students.insert(0, 'student_id', range(1, len(df) + 1))

scores = df[['math score', 'reading score', 'writing score']].copy()
scores.insert(0, 'student_id', range(1, len(df) + 1))

scores_dup = pd.concat([scores, scores.sample(5, random_state=1)], ignore_index=True)
print(len(scores), "->", len(scores_dup))

1000 -> 1005


### TASK 8: Давхардал устгах ба нэгтгэх

1. `scores_dup` дахь давхардсан `student_id`-тай мөрийг устга.
2. `students` ба `scores_clean` хүснэгтийг `student_id` дээр нэгтгэ (inner join).


In [ ]:
scores_clean = scores_dup.drop_duplicates(subset=['student_id'])
merged = students.merge(scores_clean, on='student_id', how='inner')

print(len(scores_clean), merged.shape)
merged.head()

1000 (1000, 9)


,student_id,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,1,female,group D,associate's degree,standard,completed,93,97,100
1,2,male,group B,master's degree,standard,none,72,51,63
2,3,female,group D,high school,standard,completed,53,66,64
3,4,female,group C,master's degree,standard,none,68,100,89
4,5,female,group D,bachelor's degree,standard,completed,100,100,100


### TASK 9: Redundancy – Correlation analysis

Тоон attribute-уудын хоорондын корреляцийг ол. Өндөр корреляцтай attribute нь илүүдэл (redundant) байж болно.


In [ ]:
corr = merged[['math score', 'reading score', 'writing score']].corr()
corr

,math score,reading score,writing score
math score,1.000000,0.735258,0.732973
reading score,0.735258,1.000000,0.819321
writing score,0.732973,0.819321,1.000000


### TASK 10: Nominal attribute – Chi-square тест

`gender` ба `lunch` хоёр хамааралтай эсэхийг χ² тестээр шалга.  
(p-value < 0.05 бол хамааралтай гэж үзнэ.)


In [15]:
%pip install scipy

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from scipy.stats import chi2_contingency

contingency = pd.crosstab(merged['gender'], merged['lunch'])
chi2, p, dof, expected = chi2_contingency(contingency)

print("chi2 =", round(chi2, 3), " p-value =", round(p, 4), " dof =", dof)
contingency

chi2 = 1.63  p-value = 0.2018  dof = 1


lunch,free/reduced,standard
gender,,
female,148,338
male,177,337


### TASK 10.1: Тайлбар (үгээр)

1. Аль хоёр оноо хамгийн өндөр корреляцтай вэ? Тэдгээрийн нэгийг хасаж болох уу?
2. Chi-square тестийн үр дүнгээс юу дүгнэж байна вэ?


## 5) Data Transformation – Normalization

Хэмжих нэгж, муж өөр attribute-ууд шинжилгээнд өөр жин авдаг.  
Normalization нь бүх attribute-д ижил жин өгөхийг зорино.


### TASK 11: Гурван normalization арга

`math score` дээр дараах аргуудыг хэрэглэ:

- **Min-Max:** `(v − min) / (max − min)`
- **Z-score:** `(v − mean) / std`
- **Decimal scaling:** `v / 10^j` (j нь max(|v'|) < 1 байх хамгийн бага бүхэл тоо)


In [ ]:
x = df['math score']

df['math_minmax'] = (x - x.min()) / (x.max() - x.min())
df['math_z'] = (x - x.mean()) / x.std()

j = len(str(int(x.abs().max())))
df['math_decimal'] = x / 10 ** j

df[['math score', 'math_minmax', 'math_z', 'math_decimal']].head()

,math score,math_minmax,math_z,math_decimal
0,93,0.93,1.582457,0.093
1,72,0.72,0.130979,0.072
2,53,0.53,-1.182264,0.053
3,68,0.68,-0.145493,0.068
4,100,1.00,2.066283,0.100


### TASK 12: Харьцуулалт

Дөрвөн баганын min, max, mean, std утгыг нэг хүснэгтээр харуул.


In [ ]:
cols = ['math score', 'math_minmax', 'math_z', 'math_decimal']
df[cols].describe().loc[['min', 'max', 'mean', 'std']]

,math score,math_minmax,math_z,math_decimal
min,0.000000,0.00000,-4.845518e+00,0.000000
max,100.000000,1.00000,2.066283e+00,0.100000
mean,70.105000,0.70105,-2.859935e-16,0.070105
std,14.468009,0.14468,1.000000e+00,0.014468


### TASK 12.1: Тайлбар (үгээр)

1. Аргуудын гаралтын муж хэд вэ?
2. Outlier олонтой үед аль арга илүү тохиромжтой вэ? Яагаад?
3. Шинэ өгөгдөл ирэхэд яагаад min, max (эсвэл mean, std)-ийг хадгалах хэрэгтэй вэ?


## 6) Data Transformation – Discretization

Тоон утгыг интервал эсвэл ойлголтын шошгоор (Low, Medium, High) солино.


### TASK 13: Equal-width

`math score`-ыг **ижил өргөнтэй** 3 интервалд хуваа.


In [ ]:
df['math_width'] = pd.cut(df['math score'], bins=3, labels=['Low', 'Medium', 'High'])
df['math_width'].value_counts().sort_index()

math_width
Low         8
Medium    387
High      605
Name: count, dtype: int64

### TASK 14: Equal-frequency

`math score`-ыг **ойролцоо тооны утгатай** 3 бүлэгт хуваа.


In [ ]:
df['math_freq'] = pd.qcut(df['math score'], q=3, labels=['Low', 'Medium', 'High'])
df['math_freq'].value_counts().sort_index()

math_freq
Low       348
Medium    344
High      308
Name: count, dtype: int64

### TASK 15: Concept hierarchy (өөрийн босго)

Lab 3-ын ангиллыг ашигла:  
`< 60` → Low, `60–79` → Medium, `≥ 80` → High


In [ ]:
df['math_level'] = pd.cut(df['math score'], bins=[0, 60, 80, 101],
                          labels=['Low', 'Medium', 'High'], right=False)
df['math_level'].value_counts().sort_index()

math_level
Low       228
Medium    510
High      262
Name: count, dtype: int64

### TASK 15.1: Тайлбар (үгээр)

1. Equal-width ба equal-frequency аргын bucket бүрийн тоо ямар ялгаатай байна вэ?
2. Аль арга outlier-т илүү мэдрэмтгий вэ?
3. Бодит хэрэглээнд аль аргыг хэзээ сонгох вэ?


## 7) Бүх алхмын дараалал (Concept Check)

Өгөгдөл бэлтгэх процесс: **чанар шалгах → цэвэрлэх → нэгтгэх → хувиргах**.


### TASK 16: Pipeline thinking (Verbal)

1. Дээрх 4 алхмыг яагаад энэ дарааллаар хийдэг вэ?
2. Normalization-ийг missing values нөхөхөөс ӨМНӨ хийвэл ямар асуудал гарах вэ?
3. Классификацийн lab-д орохоос өмнө өөрийн dataset дээр аль алхмуудыг заавал хийх вэ?


### Тайлбар
- 


# Data Mining – Lab 4 Тайлан
## Өгөгдөл бэлтгэх ба чанар

**Нэр:**  
**Оюутны код:**  
**Анги / Бүлэг:**  
**Огноо:**  

---

## 1. Lab зорилго (1–2 өгүүлбэр)
Энэ лабораторийн ажлын зорилгыг өөрийн үгээр тайлбарлана уу.

---

## 2. Dataset-ийн товч тайлбар
- Dataset нэр:
- Мөр, баганын тоо:
- Ашигласан үндсэн баганууд:

---

## 3. Data Quality ба Cleaning

### 3.1 Missing values
- Ямар аргуудыг хэрэглэсэн бэ (ignore, mean, median, class mean, mode)?
- Аль нь хамгийн тохиромжтой байсан бэ? Яагаад?

### 3.2 Noisy data
- Binning-ийн үр дүн (bin means / bin boundaries) юу байсан бэ?
- IQR ба z-score аргаар хэдэн outlier олсон бэ?

---

## 4. Data Integration
- Хоёр хүснэгтийг яаж нэгтгэсэн бэ?
- Хэдэн давхардсан мөр устгасан бэ?
- Correlation болон chi-square тестийн дүгнэлт:

---

## 5. Data Transformation

### 5.1 Normalization
- Min-Max, z-score, decimal scaling аргуудын ялгаа юу вэ?
- Аль аргыг хэзээ сонгох вэ?

### 5.2 Discretization
- Equal-width ба equal-frequency аргын үр дүнг харьцуулна уу.

---

## 6. Сэтгэхүй
Data preparation-ийн 4 алхмын дарааллыг яагаад чухал гэж үзэж байна вэ? (3–4 өгүүлбэр)

---

## 7. Reflection

1. Өгөгдлийн чанар муу байвал data mining-ийн үр дүнд юу болох вэ?
2. Энэ lab дараагийн classification lab-д яаж бэлтгэж өгсөн бэ?
3. Ямар хэсэг хамгийн ойлгомжгүй байсан бэ?

(3–5 өгүүлбэр)

---

## 8. Дүгнэлт

Энэ лабораториос сурсан хамгийн чухал 1–2 санааг бичнэ үү.